# SQL with SQLite — A Hands-On Notebook

This notebook teaches SQL by running real queries against a small SQLite database, built entirely with Python's built-in `sqlite3` module. **Nothing to install.**

**How to run this:**
- In VS Code: open this file, it will prompt you to install the "Jupyter" extension if you don't have it. Accept it.
- Alternative: `pip install notebook --break-system-packages` then `jupyter notebook` from the terminal.

**Rules for using this notebook:**
- Run the cells **top to bottom, in order** — later cells depend on the database built in earlier ones.
- Read the markdown before each code cell. Don't skip to the exercises.
- The last section is 5 exercises with **no solutions provided on purpose** — bring your queries back to me when you're done, or tell me which exercise number you're stuck on.

## Setup

We're using an **in-memory** SQLite database — it exists only while this notebook is running, which is perfect for practice. When you build a real project later, you'll swap `":memory:"` for a filename like `"movies.db"` and it persists to disk.

The `run_query()` helper below just makes results print as a readable table instead of a raw list of tuples.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

def run_query(query, params=()):
    """Run a query and pretty-print the results as a simple table."""
    cursor.execute(query, params)
    rows = cursor.fetchall()
    if cursor.description is None:
        conn.commit()
        print("OK — statement executed.")
        return
    headers = [d[0] for d in cursor.description]
    widths = [max(len(str(h)), max((len(str(r[i])) for r in rows), default=0)) for i, h in enumerate(headers)]
    def fmt(row):
        return " | ".join(str(v).ljust(w) for v, w in zip(row, widths))
    print(fmt(headers))
    print("-+-".join("-" * w for w in widths))
    for row in rows:
        print(fmt(row))
    print(f"\n({len(rows)} row{'s' if len(rows) != 1 else ''})")

print("Connected. Ready to go.")

## 1. Creating tables — `CREATE TABLE`

A database is just organized tables. Each table has columns with **types** and **constraints**:

- `INTEGER PRIMARY KEY` — a unique ID for each row, auto-tracked by SQLite
- `TEXT` / `INTEGER` / `REAL` — SQLite's core types (SQLite is loosely typed compared to other databases, but write correct types anyway — it matters everywhere else)
- `NOT NULL` — this column can never be empty
- `FOREIGN KEY` — a column that points to another table's primary key, linking the two

We'll build a small movie database: **directors**, **movies**, **actors**, and a **movie_actors** table that links movies to actors (a movie has many actors, an actor is in many movies — that many-to-many relationship needs its own linking table).

In [ ]:
cursor.executescript("""
CREATE TABLE directors (
    director_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    birth_year INTEGER,
    country TEXT
);

CREATE TABLE movies (
    movie_id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    release_year INTEGER,
    director_id INTEGER,
    genre TEXT,
    budget_millions REAL,
    box_office_millions REAL,
    rating REAL,
    FOREIGN KEY (director_id) REFERENCES directors(director_id)
);

CREATE TABLE actors (
    actor_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    birth_year INTEGER
);

CREATE TABLE movie_actors (
    movie_id INTEGER,
    actor_id INTEGER,
    role TEXT,
    FOREIGN KEY (movie_id) REFERENCES movies(movie_id),
    FOREIGN KEY (actor_id) REFERENCES actors(actor_id)
);
""")
conn.commit()
print("Tables created.")

## 2. Inserting data — `INSERT INTO`

Syntax: `INSERT INTO table (col1, col2, ...) VALUES (val1, val2, ...);`

Running a batch of inserts to populate the database. Notice director #6 (Kenji Watanabe) has **no movies** — that's intentional, it'll matter later when we cover `LEFT JOIN`.

In [ ]:
cursor.executescript("""
INSERT INTO directors (director_id, name, birth_year, country) VALUES
(1, 'Elena Cho', 1975, 'South Korea'),
(2, 'Marcus Webb', 1968, 'USA'),
(3, 'Priya Nair', 1982, 'India'),
(4, 'Tomas Berg', 1971, 'Sweden'),
(5, 'Aisha Rahman', 1988, 'Lebanon'),
(6, 'Kenji Watanabe', 1990, 'Japan');

INSERT INTO movies (movie_id, title, release_year, director_id, genre, budget_millions, box_office_millions, rating) VALUES
(1, 'Silent Horizon', 2014, 1, 'Sci-Fi', 45, 210, 7.8),
(2, 'The Long Way Home', 2016, 2, 'Drama', 12, 38, 8.1),
(3, 'Redline', 2018, 4, 'Action', 90, 340, 6.9),
(4, 'Paper Moons', 2019, 3, 'Drama', 8, 22, 8.4),
(5, 'Glass City', 2020, 1, 'Sci-Fi', 120, 410, 7.2),
(6, 'Nightfall', 2012, 2, 'Thriller', 25, 95, 7.5),
(7, 'Echoes', 2021, 5, 'Drama', 5, 14, 8.7),
(8, 'Static', 2017, 4, 'Thriller', 30, 88, 6.5),
(9, 'The Last Frame', 2022, 5, 'Documentary', 3, 6, 8.9),
(10, 'Firelight', 2015, 3, 'Comedy', 15, 60, 7.0),
(11, 'Wavelength', 2023, 1, 'Sci-Fi', 150, 480, 7.6),
(12, 'Borrowed Time', 2011, 2, 'Drama', 10, 33, 7.9);

INSERT INTO actors (actor_id, name, birth_year) VALUES
(1, 'Nadia Farrow', 1985), (2, 'Colin Bishop', 1979), (3, 'Renata Silva', 1990),
(4, 'Omar Haddad', 1983), (5, 'Julia Ferreira', 1992), (6, 'Marcus Lee', 1980),
(7, 'Elif Demir', 1988), (8, 'Grace Tanaka', 1995), (9, 'Victor Osei', 1987),
(10, 'Lena Fischer', 1991), (11, 'Samuel Diallo', 1984), (12, 'Priya Deshmukh', 1993),
(13, 'Hana Kobayashi', 1989), (14, 'Tariq Amin', 1981), (15, 'Sofia Moretti', 1994);

INSERT INTO movie_actors (movie_id, actor_id, role) VALUES
(1, 1, 'lead'), (1, 2, 'support'),
(2, 3, 'lead'), (2, 4, 'support'),
(3, 6, 'lead'), (3, 5, 'support'), (3, 2, 'support'),
(4, 7, 'lead'), (4, 8, 'support'),
(5, 1, 'lead'), (5, 9, 'support'),
(6, 10, 'lead'), (6, 11, 'support'),
(7, 12, 'lead'), (7, 13, 'support'),
(8, 14, 'lead'), (8, 15, 'support'), (8, 10, 'support'),
(9, 15, 'narrator'), (9, 8, 'support'),
(10, 4, 'lead'), (10, 5, 'support'),
(11, 1, 'lead'), (11, 6, 'support'),
(12, 3, 'lead'), (12, 2, 'support');
""")
conn.commit()
print("Data inserted.")

## 3. SELECT basics

- `SELECT columns FROM table` — pick what you want to see
- `WHERE condition` — filter rows
- `ORDER BY column [ASC|DESC]` — sort results
- `LIMIT n` — cap the number of rows
- `DISTINCT` — remove duplicates
- `SELECT *` — all columns (fine for exploring, avoid in real code — always name your columns)

In [ ]:
run_query("SELECT title, release_year FROM movies;")

In [ ]:
run_query("SELECT * FROM movies WHERE genre = 'Sci-Fi';")

In [ ]:
run_query("SELECT title, rating FROM movies WHERE rating > 8 ORDER BY rating DESC;")

In [ ]:
run_query("SELECT DISTINCT genre FROM movies;")

In [ ]:
run_query("SELECT title, release_year FROM movies ORDER BY release_year DESC LIMIT 3;")

## 4. Aggregate functions & GROUP BY / HAVING

- `COUNT()`, `SUM()`, `AVG()`, `MIN()`, `MAX()` — compute a single value across rows
- `GROUP BY column` — bucket rows into groups before aggregating (e.g. "per genre" instead of "across everything")
- `HAVING` — filters groups *after* aggregation. `WHERE` filters rows *before* grouping; `HAVING` filters the grouped results. This distinction trips everyone up at first — that's exactly why it matters.

In [ ]:
run_query("SELECT COUNT(*) AS total_movies FROM movies;")

In [ ]:
run_query("""
SELECT genre, AVG(rating) AS avg_rating
FROM movies
GROUP BY genre;
""")

In [ ]:
run_query("""
SELECT genre, COUNT(*) AS num_movies
FROM movies
GROUP BY genre
HAVING COUNT(*) >= 2;
""")

In [ ]:
run_query("""
SELECT director_id, SUM(box_office_millions) AS total_box_office
FROM movies
GROUP BY director_id
ORDER BY total_box_office DESC;
""")

## 5. JOINs

This is the concept that actually makes relational databases useful — combining rows from multiple tables based on a matching column.

- **INNER JOIN**: only returns rows that match in both tables
- **LEFT JOIN**: returns everything from the left table, even if there's no match on the right (unmatched columns come back as `NULL`)

In [ ]:
run_query("""
SELECT m.title, d.name AS director, d.country
FROM movies m
JOIN directors d ON m.director_id = d.director_id;
""")

In [ ]:
run_query("""
SELECT m.title, a.name AS actor, ma.role
FROM movies m
JOIN movie_actors ma ON m.movie_id = ma.movie_id
JOIN actors a ON ma.actor_id = a.actor_id
ORDER BY m.title;
""")

Now watch what `LEFT JOIN` does with Kenji Watanabe, who has zero movies. An `INNER JOIN` here would silently drop him from the results — a `LEFT JOIN` keeps him with `0` movies. This difference is the source of a huge number of "why is data missing" bugs in real applications.

In [ ]:
run_query("""
SELECT d.name, COUNT(m.movie_id) AS num_movies
FROM directors d
LEFT JOIN movies m ON d.director_id = m.director_id
GROUP BY d.director_id
ORDER BY num_movies DESC;
""")

## 6. Subqueries

A query nested inside another query — useful when you need a computed value (like an average) to filter against, or when you need to first figure out a set of IDs before filtering the main query with them.

In [ ]:
run_query("""
SELECT title, rating
FROM movies
WHERE rating > (SELECT AVG(rating) FROM movies)
ORDER BY rating DESC;
""")

In [ ]:
run_query("""
SELECT name
FROM directors
WHERE director_id IN (
    SELECT director_id FROM movies GROUP BY director_id HAVING COUNT(*) > 2
);
""")

## 7. UPDATE and DELETE

**The most important rule in this whole notebook:** always know your `WHERE` clause before you hit run. `UPDATE movies SET rating = 0;` with no `WHERE` updates every single row. `DELETE FROM movies;` with no `WHERE` empties the entire table. Both are permanent unless you're in a transaction you haven't committed.

In [ ]:
run_query("SELECT title, rating FROM movies WHERE title = 'Redline';")

run_query("UPDATE movies SET rating = 8.0 WHERE title = 'Redline';")

run_query("SELECT title, rating FROM movies WHERE title = 'Redline';")

In [ ]:
# A safe DELETE demo: insert a throwaway row, then remove it, so the real dataset stays intact.
run_query("INSERT INTO directors (director_id, name, birth_year, country) VALUES (99, 'Test Director', 2000, 'Nowhere');")
run_query("SELECT * FROM directors WHERE director_id = 99;")
run_query("DELETE FROM directors WHERE director_id = 99;")
run_query("SELECT * FROM directors WHERE director_id = 99;")

## 8. Constraints & indexes (brief — this gets its own deep dive in LU's Database courses)

- `PRIMARY KEY` — uniquely identifies each row, can't repeat, can't be null
- `FOREIGN KEY` — enforces that a value must exist in another table (keeps data consistent — you can't have a movie pointing at a director that doesn't exist)
- `NOT NULL` — column must always have a value
- `UNIQUE` — no duplicate values allowed in that column, even without being a primary key
- **Index** — a lookup structure that makes filtering/sorting on a column much faster on large tables. Irrelevant at 12 rows, essential at 12 million. Just know it exists for now.

In [ ]:
run_query("CREATE INDEX idx_movies_genre ON movies(genre);")

## Practice Exercises

Five exercises, using only what's covered above. **No solutions included on purpose** — write your query in the empty cell below each one, run it, and bring me the result (or tell me the exercise number if you get stuck and what you tried).

**1.** List the titles and release years of every movie with a rating of 8.0 or higher, ordered from newest to oldest.

**2.** For each director, show their name and the total box office (sum, in millions) of all their movies, ordered from highest total to lowest. *(Hint: you'll need a JOIN, not just GROUP BY on `movies` alone.)*

**3.** List every movie title along with its director's name and country, but only for movies released after 2010.

**4.** Show each genre and how many movies belong to it, but only include genres with at least 3 movies. *(You did something close to this above — do it again from memory, don't scroll up.)*

**5.** Find the names of actors who appear in more than one movie. *(This one's harder — think about what `movie_actors` gives you, and what GROUP BY + HAVING can do with a JOIN.)*

In [ ]:
# Exercise 1
run_query("""

""")

In [ ]:
# Exercise 2
run_query("""

""")

In [ ]:
# Exercise 3
run_query("""

""")

In [ ]:
# Exercise 4
run_query("""

""")

In [ ]:
# Exercise 5
run_query("""

""")